# 🧬 pychesca — NMR Chemical Shift Covariance Analysis

**Welcome!** This notebook lets you run CHESCA and CHESPA analyses on your NMR chemical shift data — no coding experience required.

---

### What does this tool do?

| Analysis | What it tells you |
|----------|-------------------|
| **CHESCA** | Groups NMR residues into clusters based on how correlated their chemical shift changes are across different protein states (ligands, mutations, etc.) |
| **SVD** | A dimensionality-reduction view showing how residues and states relate to each other |
| **CHESPA** | Identifies which residues move *toward* or *away* from an activated state, and by how much |

### Reference
> Boulton, S., Akimoto, M., Selvaratnam, R., Bashiri, A., & Melacini, G. (2014). A tool set to map allosteric networks through the NMR chemical shift covariance analysis. *Scientific Reports*, 4, 7306.

---

### ▶️ How to use this notebook
1. Run the **Setup** cell (Step 1) — do this once per session
2. Choose which analysis you want: **CHESCA/SVD** or **CHESPA**
3. Upload your CSV file when prompted
4. Adjust any parameters, then run the analysis cells
5. Download your results

## ⚙️ Step 1: Install pychesca

**Run this cell first every time you open the notebook.** It installs the pychesca package. This takes about 30–60 seconds.

In [ ]:
#@title ⚙️ Install pychesca (run this first!) { display-mode: "form" }
import subprocess, sys
print("Installing pychesca...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "git+https://github.com/Dan-Burns/pychesca.git", "-q"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ pychesca installed successfully!")
else:
    print("❌ Installation failed:")
    print(result.stderr)

import pychesca
print(f"   Version: {pychesca.__version__}")

---
## 📊 Analysis A: CHESCA + SVD

Use this section if you have **combined chemical shift data** across multiple protein states.

### Required CSV format

Your file must have:
- A column named **`RESI`** (residue numbers/identifiers)
- One column per perturbation **state** (e.g., `apo`, `ligand1`, `mutant_A`)
- Values are combined chemical shifts (in ppm)

```
RESI,apo,state1,state2,state3
10,0.012,0.045,0.078,0.031
11,0.003,0.009,0.011,0.007
```

> 💡 **Tip:** Need a template? Run the cell below to download an example file.

In [ ]:
#@title 📥 Download a CHESCA example CSV { display-mode: "form" }
import pandas as pd
import numpy as np

# Generate a small synthetic example
rng = np.random.default_rng(42)
n_res = 20
resis = list(range(10, 10 + n_res))
# Two clusters: residues 10-19 correlate together, 20-29 correlate together
base_a = rng.normal(0, 1, (5, 4))
base_b = rng.normal(0, 1, (5, 4))
noise = rng.normal(0, 0.05, (n_res, 4))
data = np.vstack([np.tile(base_a, (2, 1)), np.tile(base_b, (2, 1))]) + noise
data = np.abs(data) * 0.05  # keep values in realistic ppm range

example_df = pd.DataFrame(
    data,
    index=pd.Index(resis, name="RESI"),
    columns=["apo", "state1", "state2", "state3"]
)
example_df.to_csv("chesca_example.csv")
print("✅ Example file saved as 'chesca_example.csv'")
print("   You can download it from the Files panel (📁 icon on the left).")
print()
print("Preview:")
example_df.head()

### 📂 Upload your CHESCA CSV

In [ ]:
#@title 📂 Upload a CHESCA CSV file { display-mode: "form" }
from google.colab import files
import pandas as pd
import io

print("Click 'Choose Files' and select your CSV file.")
uploaded = files.upload()

if not uploaded:
    print("⚠️  No file uploaded.")
else:
    filename = list(uploaded.keys())[0]
    chesca_df = pd.read_csv(io.BytesIO(uploaded[filename]), index_col="RESI")
    print(f"\n✅ Loaded '{filename}'")
    print(f"   Residues: {len(chesca_df)}  |  States: {len(chesca_df.columns)}")
    print(f"   States: {list(chesca_df.columns)}")
    print()
    print("Preview (first 5 rows):")
    display(chesca_df.head())

### 🔧 Set CHESCA Parameters

- **Correlation cutoff (%):** Residues with correlation above this threshold are grouped into the same cluster. Higher values → stricter clustering (fewer, tighter clusters). Typical range: 94–99%.
- **Linkage method:** Controls how inter-cluster distances are measured during hierarchical clustering.
  - `complete` *(default)* — distance between the two *farthest* members of each cluster. Tends to produce compact, evenly-sized clusters.
  - `single` — distance between the two *closest* members. Can produce elongated (chained) clusters.
  - `average` — mean distance between all pairs across clusters. A middle ground.
  - `ward` — minimises total within-cluster variance. Requires Euclidean distances.
- **Minimum cluster size:** If set, only clusters with at *least* this many residues will be analyzed further (sub-cluster state analysis). Leave as 0 to skip.

In [ ]:
#@title 🔧 CHESCA Parameters { display-mode: "form" }

#@markdown **Correlation cutoff (%)** — residues above this correlation are clustered together.
correlation_cutoff = 98.0  #@param {type:"number"}

#@markdown **Linkage method** — how inter-cluster distances are measured.
linkage_method = "complete"  #@param ["complete", "single", "average", "ward"]

#@markdown **Minimum cluster size for sub-cluster analysis** — set to 0 to skip.
minimum_cluster_size = 0  #@param {type:"integer"}

#@markdown **Output folder name** (will be created automatically)
output_folder = "chesca_results"  #@param {type:"string"}

print(f"Parameters set:")
print(f"  Correlation cutoff : {correlation_cutoff}%")
print(f"  Linkage method     : {linkage_method}")
print(f"  Min cluster size   : {minimum_cluster_size if minimum_cluster_size > 0 else '(disabled)'}")
print(f"  Output folder      : {output_folder}/")

### ▶️ Run CHESCA Analysis

In [ ]:
#@title ▶️ Run CHESCA clustering + SVD { display-mode: "form" }
import os
import matplotlib
import matplotlib.pyplot as plt
from pychesca import HAC, SVD
from pychesca.plots import plot_corr, show_dendrogram, plot_svd, heatmap_correlation_cutoffs
from pychesca.visualize import clusters_to_pymol

# Make sure the data is loaded
try:
    chesca_df
except NameError:
    raise RuntimeError("⚠️ Please upload your CSV file first (run the 'Upload' cell above).")

os.makedirs(output_folder, exist_ok=True)

# --- Correlation cutoff explorer ---
print("📊 Generating correlation cutoff heatmap...")
heatmap_correlation_cutoffs(chesca_df, min_corr=90.0, save_file=f"{output_folder}/correlation_cutoff_explorer.pdf")
plt.suptitle("Pairwise Correlation Coefficients (≥ 90%) — use this to choose your cutoff", y=1.01)
plt.tight_layout()
plt.show()
print()

# --- HAC clustering ---
print(f"🔬 Running CHESCA clustering (cutoff = {correlation_cutoff}%, linkage = {linkage_method})...")
min_clust = minimum_cluster_size if minimum_cluster_size > 0 else None
if min_clust is not None:
    hac = HAC(chesca_df, cutoff=correlation_cutoff, method=linkage_method, sub_cluster_cutoff=min_clust)
else:
    hac = HAC(chesca_df, cutoff=correlation_cutoff, method=linkage_method)

print(f"   ✅ Found {hac.n_clusters} clusters across {len(chesca_df)} residues.")
print()

# --- Cluster assignment table ---
cluster_csv_path = f"{output_folder}/cluster_assignments.csv"
hac.clusters.sort_values("cluster").to_csv(cluster_csv_path)
print("📋 Cluster assignments (first 20 rows):")
display(hac.clusters.sort_values("cluster").head(20))
print(f"   Full table saved to: {cluster_csv_path}")
print()

# --- Correlation matrix ---
print("🗺️  Plotting correlation matrix...")
plot_corr(hac, cutoff=correlation_cutoff, save_file=f"{output_folder}/correlation_matrix.pdf")
plt.title(f"CHESCA Correlation Matrix (cutoff = {correlation_cutoff}%)")
plt.show()

# --- Dendrogram ---
print("🌳 Plotting dendrogram...")
show_dendrogram(hac, save_file=f"{output_folder}/dendrogram.pdf")
plt.show()

# --- SVD biplot ---
print("⚡ Running SVD...")
dims = SVD(chesca_df)
plot_svd(dims, centering="column", save_file=f"{output_folder}/svd_plot.pdf")
plt.title("SVD Biplot (column-centered) — circles = residues, diamonds = states")
plt.show()

# --- PyMOL script ---
if min_clust is not None and hasattr(hac, 'sub_cluster_ids'):
    pml_df = hac.clusters[hac.clusters["cluster"].isin(hac.sub_cluster_ids)]
    # Sub-cluster state dendrograms
    for cluster_id in hac.sub_cluster_ids:
        sub_resis = hac.clusters[hac.clusters["cluster"] == cluster_id].index
        state_corr = chesca_df.loc[sub_resis].corr().abs()
        hac_states = HAC(state_corr, cluster_states=True)
        show_dendrogram(
            hac_states,
            orientation="top",
            annotate_clusters=False,
            sub_cluster=cluster_id,
            save_file=f"{output_folder}/sub_cluster_{cluster_id}.pdf",
        )
        plt.show()
else:
    pml_df = hac.clusters

pml_path = f"{output_folder}/pymol_selections.pml"
clusters_to_pymol(pml_df, output=pml_path)
print(f"🧪 PyMOL selection script saved to: {pml_path}")
print()
print(f"✅ All done! Results saved to '{output_folder}/'")

### 💾 Download CHESCA Results

In [ ]:
#@title 💾 Download all CHESCA results as a ZIP { display-mode: "form" }
import zipfile
import os
from google.colab import files

zip_name = f"{output_folder}.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(output_folder):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            zf.write(fpath)

print(f"📦 Zipped results to '{zip_name}'")
print("⬇️  Downloading now...")
files.download(zip_name)

---
## 📊 Analysis B: CHESPA

CHESPA (**CH**emical Shift **P**rojection **A**nalysis) identifies false-positive CHESCA clusters by computing, for each residue, how far along the activation pathway it moves.

### Required CSV format

Your file must have a **`RESI`** column plus these six chemical shift columns:

| Column | Meaning |
|--------|----------|
| `refw1` | Reference state — heteronucleus (¹⁵N or ¹³C) frequency |
| `refw2` | Reference state — proton (¹H) frequency |
| `Aw1` | State A (inactive/antagonist) — heteronucleus |
| `Aw2` | State A — proton |
| `Bw1` | State B (active/agonist) — heteronucleus |
| `Bw2` | State B — proton |

```
RESI,refw1,refw2,Aw1,Aw2,Bw1,Bw2
10,120.1,8.21,120.4,8.35,119.9,8.10
```

> 💡 **Tip:** Run the cell below to download an example file.

In [ ]:
#@title 📥 Download a CHESPA example CSV { display-mode: "form" }
import subprocess
import os
from google.colab import files as colab_files

# Download the example file from the pychesca GitHub repo
url = "https://raw.githubusercontent.com/Dan-Burns/pychesca/main/chespa_cs.csv"
result = subprocess.run(["wget", "-q", url, "-O", "chespa_example.csv"], capture_output=True)

if result.returncode == 0 and os.path.getsize("chespa_example.csv") > 100:
    print("✅ Example CHESPA file saved as 'chespa_example.csv'")
    print("   You can download it from the Files panel (📁 icon on the left).")
    import pandas as pd
    preview = pd.read_csv("chespa_example.csv", index_col="RESI", nrows=5)
    print()
    print("Preview:")
    display(preview)
else:
    print("⚠️  Could not download example file from GitHub.")
    print("   Please upload your own CHESPA CSV file.")

### 📂 Upload your CHESPA CSV

In [ ]:
#@title 📂 Upload a CHESPA CSV file { display-mode: "form" }
from google.colab import files
import io

print("Click 'Choose Files' and select your CSV file.")
print("(If you downloaded the example above, you can upload 'chespa_example.csv')")
uploaded = files.upload()

if not uploaded:
    print("⚠️  No file uploaded.")
else:
    chespa_filename = list(uploaded.keys())[0]
    # Save it to disk so Chespa can read it
    chespa_local_path = f"/tmp/{chespa_filename}"
    with open(chespa_local_path, "wb") as f:
        f.write(uploaded[chespa_filename])
    print(f"\n✅ Loaded '{chespa_filename}'")
    import pandas as pd
    preview = pd.read_csv(chespa_local_path)
    print(f"   Residues: {len(preview)}")
    print()
    print("Preview (first 5 rows):")
    display(preview.head())

### 🔧 Set CHESPA Parameters

In [ ]:
#@title 🔧 CHESPA Parameters { display-mode: "form" }

#@markdown **Heteronucleus type** — the type of nucleus in your `w1` dimension.
heteronucleus = "N"  #@param ["N", "C"]

#@markdown **Output folder name** (will be created automatically)
chespa_output_folder = "chespa_results"  #@param {type:"string"}

het_label = "¹⁵N" if heteronucleus == "N" else "¹³C"
print(f"Parameters set:")
print(f"  Heteronucleus : {het_label}  (scaling coefficient: {'0.2' if heteronucleus == 'N' else '0.25'})")
print(f"  Output folder : {chespa_output_folder}/")

### ▶️ Run CHESPA Analysis

In [ ]:
#@title ▶️ Run CHESPA analysis { display-mode: "form" }
import os
import pandas as pd
import matplotlib.pyplot as plt
from pychesca import Chespa
from pychesca.plots import plot_chespa

# Make sure data is loaded
try:
    chespa_local_path
except NameError:
    raise RuntimeError("⚠️ Please upload your CSV file first (run the 'Upload' cell above).")

os.makedirs(chespa_output_folder, exist_ok=True)

print(f"🔬 Running CHESPA analysis (heteronucleus: {heteronucleus})...")
chespa = Chespa(chespa_local_path, het_nuc=heteronucleus)
print(f"   ✅ Analyzed {len(chespa.resis)} residues.")
print()

# --- Results table ---
results_df = pd.DataFrame({
    "RESI": chespa.resis,
    "delta_delta_comb": chespa.ref_to_A,
    "cos_theta": chespa.cos_theta,
    "X_fractional_activation": chespa.X,
}).set_index("RESI")

results_csv_path = f"{chespa_output_folder}/chespa_results.csv"
results_df.to_csv(results_csv_path)

print("📋 Results (first 10 residues):")
display(results_df.head(10))
print()
print("Column meanings:")
print("  delta_delta_comb     — combined chemical shift distance from ref → state A (ppm)")
print("  cos_theta            — cos θ: how aligned the A shift is with the B (activation) direction")
print("                          +1 = perfectly aligned, −1 = opposite, 0 = perpendicular")
print("  X_fractional_activation — fractional activation: >0 → toward active, <0 → toward inactive")
print()

# --- Plot ---
print("📈 Generating CHESPA bar plots...")
chespa_plot_path = f"{chespa_output_folder}/chespa_plots.pdf"
plot_chespa(chespa, save_file=chespa_plot_path)
plt.show()

print(f"\n✅ All done! Results saved to '{chespa_output_folder}/'")

### 🔍 Highlight Key Residues

In [ ]:
#@title 🔍 Show residues with strongest / weakest activation { display-mode: "form" }

#@markdown Show the top N residues sorted by **fractional activation (X)**
top_n = 10  #@param {type:"integer"}

#@markdown Minimum |cos θ| to consider (filter out ambiguous directions)
min_cos_theta = 0.5  #@param {type:"number"}

try:
    results_df
except NameError:
    raise RuntimeError("⚠️  Please run the CHESPA analysis cell first.")

filtered = results_df[results_df["cos_theta"].abs() >= min_cos_theta].copy()
print(f"Residues with |cos θ| ≥ {min_cos_theta}: {len(filtered)} (out of {len(results_df)})")
print()

print(f"🔴 Top {top_n} residues moving TOWARD activation (X > 0):")
toward = filtered.sort_values("X_fractional_activation", ascending=False).head(top_n)
display(toward)
print()

print(f"🔵 Top {top_n} residues moving AWAY from activation (X < 0):")
away = filtered.sort_values("X_fractional_activation", ascending=True).head(top_n)
display(away)

### 💾 Download CHESPA Results

In [ ]:
#@title 💾 Download all CHESPA results as a ZIP { display-mode: "form" }
import zipfile
import os
from google.colab import files

zip_name = f"{chespa_output_folder}.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(chespa_output_folder):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            zf.write(fpath)

print(f"📦 Zipped results to '{zip_name}'")
print("⬇️  Downloading now...")
files.download(zip_name)

---
## ❓ Help & Troubleshooting

| Problem | Solution |
|---------|----------|
| `NameError: chesca_df` | Run the **Upload** cell first |
| `KeyError: 'RESI'` | Make sure your CSV has a column named exactly `RESI` |
| Only 1 cluster found | Lower the **correlation cutoff** (e.g., try 95% or 90%) |
| Too many clusters | Raise the **correlation cutoff** (e.g., try 99%) |
| CHESPA: all X ≈ 0 | Check that states A and B have meaningfully different shifts from the reference |
| Session disconnected | Re-run **Step 1** (install cell), then re-upload your file |

### 📧 Contact / Source Code
- GitHub: [Dan-Burns/pychesca](https://github.com/Dan-Burns/pychesca)
- PyPI: [pychesca](https://pypi.org/project/pychesca/)

### 📖 Citation
> Boulton, S., Akimoto, M., Selvaratnam, R., Bashiri, A., & Melacini, G. (2014). A tool set to map allosteric networks through the NMR chemical shift covariance analysis. *Scientific Reports*, **4**, 7306. https://doi.org/10.1038/srep07306